## Задание 1. 
Используя уравнения фильтра Калмана определите «точные координаты» разгрузки самосвала. 

In [69]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math
import pandas as pd
import numpy as np
import datetime as dt
import sys
import configparser
import os

pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

mainpath=os.path.abspath(os.getcwd())
print(mainpath)
os.chdir(mainpath)

c:\Users\Amega\barnacle\fantastic-barnacle\DataAnalysis\lab3\route4


In [74]:
data = pd.read_csv(mainpath+'\\4_w.csv', sep=';',encoding='cp1251')

# display(data.where(data['Статус самосвала'] == 'Груженный стоит'))
display(data.head(3))

,Unnamed: 0,len,Скорость,Широта,Долгота,Вес мгновенный,Продольный наклон,Поперечный наклон,Высота,"Уровень топлива, литры","Уровень топлива, проценты",путь,ДВС.Расход топлива,Ускорение по оси X,Ускорение по оси Y,Ускорение по оси Z,Статус самосвала,Самосвал,Модель самосвала,Грузоподъемность,Обороты двигателя,Расчетный выходной крутящий момент двигателя (в процентах от эталонного),Текущая нагрузка двигателя,Давление в левой передней шине,Температура в левой передней шине,Давление в правой передней шине,Температура в правой передней шине,Давление в левой задней внешней шине,Температура в левой задней внешней шине,Давление в левой задней внутренней шине,Температура в левой задней внутренней шине,Давление в правой задней внутренней шине,Температура в правой задней внутренней шине,Давление в правой задней внешней шине,Температура в правой задней внешней шине,Расход топлива,Давление левый передний цилиндр,Давление правый передний цилиндр,Давление левый задний цилиндр,Давление правый задний цилиндр,уклон,azimut,isin_border,Давление левый передний цилиндр_,Давление правый передний цилиндр_,Давление левый задний цилиндр_,Давление правый задний цилиндр_,point,nom_point,"уровень давления впереди,МПа","уровень давления сзади,МПа",рекомендованная скорость_turns,Поворот влево,Поворот вправо,centr,radius,подъем,спуск,рекомендованная скорость_min,рекомендованная скорость_max
0,0,0.0,8.0,53.723291,86.837378,190.0,3.410465,1.356646,1.192848,1475.0,52.8,0.000000,150.475,-0.007648,-1.009483,-0.060161,Груженный движется вперед,95,7530G,240,1214.0,49.5,69.5,788.0,29.0,788.0,28.0,860.0,42.0,860.0,38.0,8.0,40.0,832.0,43.0,0.208993,5.9,6.6,11.4,12.9,2.106104,-2.253108,0,6.6,6.60,10.60,10.6,-,0,6.6,10.6,0.0,0,0,-,0.0,0,0,0.0,0.0
1,1,10.0,10.0,53.723223,86.837283,186.0,1.844233,0.772100,0.558264,1475.0,53.2,9.839546,198.225,0.084633,-0.946263,-0.030590,Груженный движется вперед,95,7530G,240,1237.0,65.0,99.0,788.0,29.0,788.0,28.0,860.0,42.0,860.0,38.0,8.0,40.0,832.0,43.0,0.429243,5.7,11.0,14.3,12.8,1.753915,-2.359241,0,6.6,11.55,15.37,10.6,point_60%,1,6.6,10.6,0.0,0,0,-,0.0,0,0,0.0,0.0
2,2,21.0,8.0,53.723150,86.837159,190.0,3.079763,0.208071,0.162660,1475.0,55.2,11.543509,279.375,-0.036708,-1.022739,-0.055063,Груженный движется вперед,95,7530G,240,1393.0,81.0,99.5,788.0,29.0,788.0,28.0,860.0,42.0,860.0,38.0,8.0,40.0,832.0,43.0,0.662056,6.3,6.2,9.7,11.9,1.519741,-2.265271,0,6.6,6.60,15.37,10.6,point_60%,1,6.6,10.6,0.0,0,0,-,0.0,0,0,0.0,0.0


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
import os
import sys
from itertools import product

# ===============================
# 1. Загрузка данных для всех дорог (фиксированный путь)
# ===============================
def load_truck_data(filepath, encoding='cp1251'):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Файл {filepath} не найден.")
    
    df = pd.read_csv(filepath, sep=';', encoding=encoding, decimal='.')
    
    lat_col = None
    lon_col = None
    status_col = None
    speed_col = None
    for col in df.columns:
        col_low = col.lower()
        if 'широта' in col_low:
            lat_col = col
        elif 'долгота' in col_low:
            lon_col = col
        elif 'статус' in col_low:
            status_col = col
        elif 'скорость' in col_low:
            speed_col = col
    
    
    lat = pd.to_numeric(df[lat_col], errors='coerce')
    lon = pd.to_numeric(df[lon_col], errors='coerce')
    speed = pd.to_numeric(df[speed_col], errors='coerce')
    status = df[status_col].astype(str).str.strip()
    
    df_clean = pd.DataFrame({
        'широта': lat,
        'долгота': lon,
        'статус': status,
        'скорость': speed,
        'временной_индекс': np.arange(len(df))
    })
    
    df_clean = df_clean.dropna(subset=['широта', 'долгота', 'скорость'])
    return df_clean

def load_all_routes(base_dir, file_pattern='{}_w.csv', encoding='cp1251'):
    routes = ['Д1', 'Д2', 'Д3', 'Д4']
    data = {}
    for i, route in enumerate(routes, start=1):
        filename = file_pattern.format(i)
        filepath = os.path.join(base_dir, filename)
        if not os.path.exists(filepath):
            print(f"Предупреждение: файл {filepath} не найден, пропускаем")
            continue
        try:
            df = load_truck_data(filepath, encoding=encoding)
            print(f"Загружена дорога {route} из {filepath}: {len(df)} записей")
            data[route] = df
        except Exception as e:
            print(f"Ошибка при загрузке {route}: {e}")
    return data

# ===============================
# 2. Перевод градусов в метры
# ===============================
def lat_lon_to_xy(df):
    ref_lat = df['широта'].iloc[0]
    ref_lon = df['долгота'].iloc[0]
    ref_lat_rad = np.radians(ref_lat)
    m_per_deg_lat = 111111 
    m_per_deg_lon = 111111 * np.cos(ref_lat_rad)
    df = df.copy()
    df['x'] = (df['долгота'] - ref_lon) * m_per_deg_lon
    df['y'] = (df['широта'] - ref_lat) * m_per_deg_lat
    return df

# ===============================
# 3. Фильтр Калмана (раздельные σ_x, σ_y)
# ===============================
class KalmanFilter:
    def __init__(self, dt, sigma_a=0.5, sigma_x=5.0, sigma_y=5.0):
        self.dt = dt
        self.F = np.array([[1, dt, 0, 0],
                           [0, 1, 0, 0],
                           [0, 0, 1, dt],
                           [0, 0, 0, 1]])
        self.G = np.array([[dt**2/2, 0],
                           [dt, 0],
                           [0, dt**2/2],
                           [0, dt]])
        self.H = np.array([[1, 0, 0, 0],
                           [0, 0, 1, 0]])
        sigma_a2 = sigma_a**2
        q11 = (dt**4)/4 * sigma_a2
        q12 = (dt**3)/2 * sigma_a2
        q22 = dt**2 * sigma_a2
        self.Q = np.array([[q11, q12, 0, 0],
                           [q12, q22, 0, 0],
                           [0, 0, q11, q12],
                           [0, 0, q12, q22]])
        self.R = np.array([[sigma_x**2, 0],
                           [0, sigma_y**2]])
        self.x_est = np.zeros(4)
        self.P = np.eye(4) * 100.0

    def predict(self, u=np.zeros(2)):
        self.x_pred = self.F @ self.x_est + self.G @ u
        self.P_pred = self.F @ self.P @ self.F.T + self.Q
        return self.x_pred

    def update(self, z):
        S = self.H @ self.P_pred @ self.H.T + self.R
        K = self.P_pred @ self.H.T @ np.linalg.inv(S)
        self.x_est = self.x_pred + K @ (z - self.H @ self.x_pred)
        self.P = (np.eye(4) - K @ self.H) @ self.P_pred
        return self.x_est, K

def apply_kalman(df, dt=1.0, sigma_a=0.5, sigma_x=5.0, sigma_y=5.0):
    kf = KalmanFilter(dt, sigma_a, sigma_x, sigma_y)
    n = len(df)
    x_k = np.zeros(n)
    y_k = np.zeros(n)
    for i in range(n):
        z = np.array([df.loc[i, 'x'], df.loc[i, 'y']])
        kf.predict()
        x_est, _ = kf.update(z)
        x_k[i] = x_est[0]
        y_k[i] = x_est[2]
    df = df.copy()
    df['x_kalman'] = x_k
    df['y_kalman'] = y_k
    return df

# ===============================
# Задание 1: фильтр Калмана
# ===============================
def task1(df, route_name):
    print(f"\n--- Дорога {route_name} ---")
    df_xy = lat_lon_to_xy(df)
    df_filt = apply_kalman(df_xy, dt=1.0, sigma_a=0.5, sigma_x=5.0, sigma_y=5.0)
    unloading_mask = (df_filt['статус'].str.contains('стоит', na=False)) & (df_filt['скорость'] == 0)
    unloading = df_filt[unloading_mask]
    if len(unloading) == 0:
        print(f"  Нет событий разгрузки для {route_name}")
        return None
    var_x = unloading['x_kalman'].var()
    var_y = unloading['y_kalman'].var()
    if np.isnan(var_x):
        var_x = 0.0
    if np.isnan(var_y):
        var_y = 0.0
    print(f"Дисперсия разгрузки: σ²(x) = {var_x:.2f} м², σ²(y) = {var_y:.2f} м²")
    print(f"Средние координаты: x = {unloading['x_kalman'].mean():.2f} м, y = {unloading['y_kalman'].mean():.2f} м")
    return var_x, var_y

# ===============================
# Задание 2: максимальное количество самосвалов
# ===============================
def compute_cycle_times():
    cycles = {
        'Д1': {'load': 2, 'loaded': 15, 'unload': 5, 'empty': 15-2},
        'Д2': {'load': 2, 'loaded': 10, 'unload': 5, 'empty': 10-2},
        'Д3': {'load': 2, 'loaded': 20, 'unload': 5, 'empty': 20-2},
        'Д4': {'load': 2, 'loaded': 15, 'unload': 5, 'empty': 15-2}
    }
    for key, val in cycles.items():
        val['total'] = val['load'] + val['loaded'] + val['unload'] + val['empty']
    return cycles

def max_trucks_per_road(wait_limit=5.0):
    cycles = compute_cycle_times()
    S = 5.0
    max_x = {}
    print("\n=== Задание 2 ===")
    for road, data in cycles.items():
        T = data['total']
        a = S / T
        x_max_float = (2 * wait_limit) / (a * (S + 2 * wait_limit)) if a != 0 else 0
        max_x[road] = int(np.floor(x_max_float))
        print(f"Максимум на {road}: {max_x[road]} самосвалов (простой ≤ {wait_limit} мин)")
    return max_x

# ===============================
# Задание 3: оптимизация количества самосвалов
# ===============================
def task3(truck, third_index=0, max_x_dict=None, total_trucks_limit=30):
    склады = {
        'Д1': {'склад': 'C', 'Ni': 2.28, 'Cu': 2.36, 'план': [2400, 1200, 3300]},
        'Д2': {'склад': 'К_б', 'Ni': 1.8, 'Cu': 1.9, 'план': [500, 500, 410]},
        'Д3': {'склад': 'К_м', 'Ni': 0.56, 'Cu': 1.17, 'план': [450, 550, 550]},
        'Д4': {'склад': 'Т', 'Ni': 1.98, 'Cu': 2.82, 'план': [1600, 1900, 2300]}
    }
    target = {
        0: {'Ni': 1.58, 'Cu': 2.66},
        1: {'Ni': 1.58, 'Cu': 2.44},
        2: {'Ni': 1.57, 'Cu': 2.58}
    }
    plan = {road: data['план'][third_index] for road, data in склады.items()}
    target_Ni = target[third_index]['Ni']
    target_Cu = target[third_index]['Cu']
    
    cycles = compute_cycle_times()
    T_cycle = {road: data['total'] for road, data in cycles.items()}
    prod_one = {road: truck * (60 / T_cycle[road]) for road in T_cycle}
    
    if max_x_dict is None:
        max_x_dict = {road: 30 for road in T_cycle}
    
    roads = ['Д1', 'Д2', 'Д3', 'Д4']
    best_x = None
    best_obj = float('inf')
    best_metrics = {}
    
    ranges = [range(0, max_x_dict[r] + 1) for r in roads]
    for x1, x2, x3, x4 in product(*ranges):
        x = [x1, x2, x3, x4]
        if sum(x) > total_trucks_limit:
            continue
        supply = {road: prod_one[road] * x[i] * 8 for i, road in enumerate(roads)}
        if not all(supply[road] >= plan[road] - 1e-5 for road in roads):
            continue
        total_tonnage = sum(prod_one[road] * x[i] for i, road in enumerate(roads))
        if total_tonnage <= 0:
            continue
        Ni_mix = sum(prod_one[road] * x[i] * (склады[road]['Ni']/100) for i, road in enumerate(roads)) / total_tonnage * 100
        Cu_mix = sum(prod_one[road] * x[i] * (склады[road]['Cu']/100) for i, road in enumerate(roads)) / total_tonnage * 100
        
        if third_index == 0:
            obj = (Ni_mix - target_Ni)**2 + (Cu_mix - target_Cu)**2
        elif third_index == 1:
            obj = (Ni_mix - target_Ni)**2
        else:
            obj = (Cu_mix - target_Cu)**2
        
        if obj < best_obj - 1e-6:
            best_obj = obj
            best_x = x.copy()
            best_metrics = {
                'Ni_mix': Ni_mix,
                'Cu_mix': Cu_mix,
                'total_tonnage': total_tonnage,
                'supply': supply.copy()
            }
    
    if best_x is None:
        print("Не найдено комбинаций, удовлетворяющих плану.")
        return None
    
    print("\n=== Задание 3 ===")
    print(f"Треть списка: {third_index+1} (план смены {third_index+1})")
    print(f"Целевое содержание: Ni={target_Ni}%, Cu={target_Cu}%")
    print(f"Грузоподъемность самосвала = {truck} т")
    for i, road in enumerate(roads):
        print(f"  {road}: {best_x[i]} шт.")
    print(f"Суммарно: {sum(best_x)} самосвалов (≤ {total_trucks_limit})")
    print(f"Достигнутое содержание: Ni={best_metrics['Ni_mix']:.3f}%, Cu={best_metrics['Cu_mix']:.3f}%")
    print(f"Отклонение: ΔNi={abs(best_metrics['Ni_mix']-target_Ni):.3f}%, ΔCu={abs(best_metrics['Cu_mix']-target_Cu):.3f}%")
    return best_x, best_metrics

# ===============================
# Основная программа
# ===============================
if __name__ == "__main__":
    base_dir = "c:/Users/Amega/barnacle/fantastic-barnacle/DataAnalysis/lab3/route4"
    routes_data = load_all_routes(base_dir, file_pattern='{}_w.csv', encoding='cp1251')
    
    
    print("\n" + "="*50)
    print("ЗАДАНИЕ 1: Фильтр Калмана и дисперсия разгрузки")
    print("="*50)
    for route, df in routes_data.items():
        task1(df, route)
    
    # Задание 2
    print("\n" + "="*50)
    print("ЗАДАНИЕ 2: максимизация")
    print("="*50)
    max_limits = max_trucks_per_road(wait_limit=5.0)
    
    груз = 50.0  # значение по умолчанию
    try:
        any_df = next(iter(routes_data.values()))
        # Пытаемся найти исходный CSV, чтобы прочитать грузоподъемность
        for route, df in routes_data.items():
            if 'Грузоподъемность' in df.columns:
                груз = float(df['Грузоподъемность'].iloc[0])
                print(f"\nГрузоподъемность из данных: {груз} т")
                break
    except Exception as e:
        print(f"\nНе удалось определить грузоподъемность, используется {груз} т")
    
    # Задание 3
    print("\n" + "="*50)
    print("ЗАДАНИЕ 3: Оптимизация количества самосвалов")
    print("="*50)
    for third in range(3):
        task3(груз, third_index=third, max_x_dict=max_limits, total_trucks_limit=30)

Загружена дорога Д1 из c:/Users/Amega/barnacle/fantastic-barnacle/DataAnalysis/lab3/route4\1_w.csv: 210 записей
Загружена дорога Д2 из c:/Users/Amega/barnacle/fantastic-barnacle/DataAnalysis/lab3/route4\2_w.csv: 206 записей
Загружена дорога Д3 из c:/Users/Amega/barnacle/fantastic-barnacle/DataAnalysis/lab3/route4\3_w.csv: 196 записей
Загружена дорога Д4 из c:/Users/Amega/barnacle/fantastic-barnacle/DataAnalysis/lab3/route4\4_w.csv: 199 записей

ЗАДАНИЕ 1: Фильтр Калмана и дисперсия разгрузки

--- Дорога Д1 ---
Дисперсия разгрузки: σ²(x) = 203109.11 м², σ²(y) = 64814.95 м²
Средние координаты: x = -5.91 м, y = 3247.84 м

--- Дорога Д2 ---
Дисперсия разгрузки: σ²(x) = 0.00 м², σ²(y) = 0.00 м²
Средние координаты: x = 720.34 м, y = 2894.49 м

--- Дорога Д3 ---
Дисперсия разгрузки: σ²(x) = 930323.51 м², σ²(y) = 165500.71 м²
Средние координаты: x = 54.36 м, y = 2636.65 м

--- Дорога Д4 ---
Дисперсия разгрузки: σ²(x) = 0.00 м², σ²(y) = 0.00 м²
Средние координаты: x = 1462.64 м, y = -3142.40 м
